# SENTINEL — Train hành vi "đánh nhau" (luồng B: khung xương + ST-GCN)

Pipeline: tải dataset → loại video trùng → trích khung xương YOLOv8n-pose ở imgsz 1280 + BoT-SORT (10 FPS) → train ST-GCN theo cặp người (MIL) → đánh giá trên tập test chuẩn.

- **Bộ chính:** UBI-Fights (1.000 video, nhãn theo frame, có tập test chuẩn `test_videos.csv`).
- **Test chéo:** Surveillance Camera Fight (300 clip, MIT).
- **Tùy chọn:** Real Life Violence Situations (Kaggle) làm dữ liệu train thêm.

Mọi bước đều chạy tiếp được sau khi Colab ngắt: khung xương lưu từng video lên Drive, train lưu checkpoint mỗi epoch. Chỉ lưu khung xương và model lên Drive, không lưu video đã giải nén.

**GPU:** `Runtime > Change runtime type > L4 GPU`. Trích khung xương chạy song song nhiều tiến trình trên cùng GPU (số tiến trình theo số CPU của máy Colab).

**Khối lượng:** chỉ trích phần cần — train: đoạn đánh nhau ±10 s và 20 s mỗi video bình thường; test: 40 video đánh nhau đầy đủ và 60 s mỗi video bình thường — khoảng 9,7 giờ video thay vì 79 giờ. Tập test rút gọn này là tập đánh giá của đề tài (không dùng để so sánh với bài báo khác).

**Treo máy:** giữ `AUTO_DISCONNECT = True` (mặc định) rồi `Runtime > Run all`. Chạy xong — hoặc gặp lỗi ở bất kỳ mục nào — notebook ghi lý do vào `MyDrive/SENTINEL/artifacts/fight/run_log.txt`, đồng bộ Drive rồi tự ngắt runtime để không tốn compute unit. Giữ tab trình duyệt mở: Colab Pro không chạy nền khi đóng tab.

## 0. Cấu hình

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/TranDuon/anomaly-behavior-detector-system.git"
BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive/SENTINEL")
REPO_DIR = DRIVE_ROOT / "repo" / "anomaly-behavior-detector-system"
ZIP_DIR = DRIVE_ROOT / "datasets" / "fight"          # chỉ giữ file zip gốc
POSE_ROOT = DRIVE_ROOT / "features" / "pose_v1"       # khung xương đã trích
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "fight"
LOCAL_DATA = Path("/content/datasets")                 # video giải nén (mất khi hết phiên)

UBI_URL = "https://socia-lab.di.ubi.pt/EventDetection/UBI_FIGHTS.zip"
UBI_ZIP_BYTES = 8_117_112_419
SCF_URL = "https://github.com/sayibet/fight-detection-surv-dataset.git"
KEEP_UBI_ZIP_ON_DRIVE = True   # False: tải lại mỗi phiên, tiết kiệm 8,1 GB Drive
USE_RLVS = False               # True: cần kaggle.json (xem mục 4b)
FIGHT_MARGIN_SECONDS = 10      # train, video đánh nhau: chỉ giữ đoạn đánh nhau ±10 s (6,0 h -> 3,5 h)
MAX_SECONDS_NORMAL = 20        # train, video bình thường: 20 s đầu mỗi video (757 video -> tối đa 4,2 h)
TEST_NORMAL_SECONDS = 60       # test, 27 video bình thường: 60 s đầu mỗi video (11,4 h -> 0,45 h)
# Thử nghiệm còn nhanh hơn: FIGHT_MARGIN_SECONDS = 5, MAX_SECONDS_NORMAL = 10 (tổng ~6,5 h video)
EVAL_STRIDE = 3                # chấm 1 cửa sổ mỗi 0,3 giây; frame ở giữa được nội suy
AUTO_DISCONNECT = True         # treo máy: chạy xong hoặc gặp lỗi thì ghi log, đồng bộ Drive rồi ngắt runtime

UBI_ROOT = LOCAL_DATA / "UBI_FIGHTS"
SCF_ROOT = LOCAL_DATA / "fight-detection-surv-dataset"
RLVS_ROOT = LOCAL_DATA / "rlvs"
print("Pose cache:", POSE_ROOT)
print("Artifacts:", ARTIFACT_DIR)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for path in [REPO_DIR.parent, ZIP_DIR, POSE_ROOT, ARTIFACT_DIR, LOCAL_DATA]:
    path.mkdir(parents=True, exist_ok=True)

import time
from contextlib import contextmanager
RUN_LOG = ARTIFACT_DIR / "run_log.txt"

def finish_and_disconnect(message):
    # Ghi log lên Drive; khi AUTO_DISCONNECT thì đồng bộ Drive rồi ngắt runtime để dừng tốn compute unit.
    with open(RUN_LOG, "a", encoding="utf-8") as log:
        log.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {message}\n")
    print(message)
    if AUTO_DISCONNECT:
        from google.colab import runtime
        drive.flush_and_unmount()
        runtime.unassign()

@contextmanager
def step(name):
    try:
        yield
    except Exception as error:
        finish_and_disconnect(f"LỖI ở mục {name}: {error!r} — xem output của mục này, sửa rồi chạy lại.")
        raise

print("Log chạy:", RUN_LOG, "| AUTO_DISCONNECT =", AUTO_DISCONNECT)

## 2. Clone lần đầu / đồng bộ code mới nhất
GitHub là nguồn chuẩn: cell bỏ mọi thay đổi trong file tracked trên Drive rồi đồng bộ với `origin`.

In [ ]:
with step('2. Đồng bộ code'):
    import os, subprocess

    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "log", "--oneline", "-1"], check=True)

## 3. Cài dependency và kiểm tra GPU

In [ ]:
with step('3. Cài dependency'):
    import sys, torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    CPUS = os.cpu_count() or 2
    POSE_SHARDS = max(1, min(6, CPUS - 1))   # tiến trình trích khung xương chạy song song
    NUM_WORKERS = max(1, CPUS - 1)           # tiến trình dựng cửa sổ khi train
    print("PyTorch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "KHÔNG CÓ GPU")
    print(f"CPU: {CPUS} | pose shards: {POSE_SHARDS} | train workers: {NUM_WORKERS}")
    (REPO_DIR / ".env").write_text(f"YOLO_DEVICE={DEVICE}\n", encoding="utf-8")

## 4. Tải dataset

**UBI-Fights:** chứng chỉ SSL của trang tác giả đã hết hạn nên dùng `--no-check-certificate`. File zip 8,1 GB; `-c` cho phép tải tiếp khi bị ngắt.

In [ ]:
with step('4. Tải dataset'):
    ubi_zip = (ZIP_DIR if KEEP_UBI_ZIP_ON_DRIVE else Path("/content")) / "UBI_FIGHTS.zip"
    if not ubi_zip.exists() or ubi_zip.stat().st_size != UBI_ZIP_BYTES:
        subprocess.run(["wget", "-c", "--no-check-certificate", "-q", "--show-progress", UBI_URL, "-O", str(ubi_zip)], check=True)
    assert ubi_zip.stat().st_size == UBI_ZIP_BYTES, "File zip UBI-Fights chưa tải đủ"
    if not (UBI_ROOT / "test_videos.csv").exists():
        subprocess.run(["unzip", "-q", "-o", str(ubi_zip), "-d", str(LOCAL_DATA)], check=True)
    print("UBI-Fights videos:", len(list(UBI_ROOT.glob("videos/*/*.mp4"))))

    if not SCF_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", SCF_URL, str(SCF_ROOT)], check=True)
    print("Surveillance Camera Fight videos:", len(list(SCF_ROOT.rglob("*.mp4"))) + len(list(SCF_ROOT.rglob("*.avi"))))

### 4b. (Tùy chọn) Real Life Violence Situations
Tải `kaggle.json` từ trang tài khoản Kaggle, upload vào `MyDrive/SENTINEL/kaggle.json`, rồi đặt `USE_RLVS = True`. Dataset chưa ghi giấy phép rõ ràng, tác giả yêu cầu trích dẫn; chỉ dùng cho nghiên cứu.

In [ ]:
with step('4b. RLVS'):
    DATASETS_TRAIN = [f"ubi:{UBI_ROOT}"]
    if USE_RLVS:
        kaggle_dir = Path.home() / ".kaggle"; kaggle_dir.mkdir(exist_ok=True)
        subprocess.run(["cp", str(DRIVE_ROOT / "kaggle.json"), str(kaggle_dir / "kaggle.json")], check=True)
        os.chmod(kaggle_dir / "kaggle.json", 0o600)
        if not RLVS_ROOT.exists():
            subprocess.run(["kaggle", "datasets", "download", "-d", "mohamedmustafa/real-life-violence-situations-dataset",
                            "-p", str(RLVS_ROOT), "--unzip"], check=True)
        DATASETS_TRAIN.append(f"rlvs:{RLVS_ROOT}")
    DATASETS_TEST = [f"ubi:{UBI_ROOT}", f"scf:{SCF_ROOT}"]
    DATASETS_ALL = DATASETS_TRAIN + [f"scf:{SCF_ROOT}"]
    print(DATASETS_ALL)

## 5. Smoke test
Trích khung xương 3 video test để chắc chắn YOLOv8-pose nhận ra người trước khi chạy toàn bộ.

In [ ]:
with step('5. Smoke test'):
    def run(script, *args):
        command = [sys.executable, "-u", script, *[str(a) for a in args]]
        subprocess.run(command, cwd=REPO_DIR, check=True)

    def dataset_args(specs):
        return [part for spec in specs for part in ("--dataset", spec)]

    smoke_root = Path("/content/pose_smoke")
    run("scripts/extract_pose.py", *dataset_args([f"ubi:{UBI_ROOT}"]), "--pose-root", smoke_root,
        "--split", "test", "--limit", "3", "--device", DEVICE, "--force")
    import json
    videos = json.loads((smoke_root / "ubi" / "manifest.json").read_text())["videos"]
    for key, info in videos.items():
        print(f"{key}: {info['samples']} mẫu, {info['person_rows'] / max(info['samples'], 1):.1f} người/mẫu, {info['tracks']} track")
    assert all(info["person_rows"] > 0 for info in videos.values()), "Không phát hiện được người — kiểm tra lại model/imgsz"

## 6. Loại video trùng giữa các dataset
Chỉ cặp khác dataset bị loại khỏi train; cặp trong cùng dataset chỉ ghi vào báo cáo để giữ tập chia chuẩn của tác giả.

In [ ]:
with step('6. Loại video trùng'):
    DEDUP_DIR = ARTIFACT_DIR / "dedup"
    run("scripts/dedup_videos.py", *dataset_args(DATASETS_ALL), "--output-dir", DEDUP_DIR)
    report = json.loads((DEDUP_DIR / "exclude.json").read_text())
    print("Cặp trùng khác dataset:", len(report["duplicate_pairs"]))
    print("Loại khỏi train:", report["exclude_from_training"][:20])

## 7. Trích khung xương (song song)
`POSE_SHARDS` tiến trình cùng dùng GPU (FP16), mỗi tiến trình nhận một phần danh sách video và ghi log riêng ở `/content/pose_shard*.log`. Chỉ trích những đoạn cần: video train đánh nhau giữ đoạn đánh nhau ±`FIGHT_MARGIN_SECONDS` giây, video train bình thường lấy `MAX_SECONDS_NORMAL` giây đầu, video test bình thường lấy `TEST_NORMAL_SECONDS` giây đầu. Video gốc không bị sửa. Chạy lại cell này sau khi Colab ngắt: video đã có cache được bỏ qua.

In [ ]:
with step('7. Trích khung xương'):
    import time
    extract_options = ["--fight-margin-seconds", str(FIGHT_MARGIN_SECONDS), "--max-seconds-normal", str(MAX_SECONDS_NORMAL),
                       "--max-seconds-test-normal", str(TEST_NORMAL_SECONDS)]
    processes = []
    for shard in range(POSE_SHARDS):
        log = open(f"/content/pose_shard{shard}.log", "w")
        command = [sys.executable, "-u", "scripts/extract_pose.py", *dataset_args(DATASETS_ALL),
                   "--pose-root", str(POSE_ROOT), "--split", "all", "--device", DEVICE,
                   *extract_options, "--shard", f"{shard}/{POSE_SHARDS}"]
        processes.append((subprocess.Popen(command, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT), log))

    def last_line(shard):
        lines = Path(f"/content/pose_shard{shard}.log").read_text(errors="ignore").strip().splitlines()
        return lines[-1] if lines else "(chưa có output)"

    started = time.time()
    while any(process.poll() is None for process, _ in processes):
        time.sleep(120)
        done = sum(1 for _ in POSE_ROOT.glob("*/*.npz")) - sum(1 for _ in POSE_ROOT.glob("*/*.partial.npz"))
        print(f"[{(time.time() - started) / 60:.0f} phút] {done} video đã có khung xương")
        for shard in range(POSE_SHARDS):
            print(f"   shard {shard}: {last_line(shard)[:110]}")
    for shard, (process, log) in enumerate(processes):
        log.close()
        if process.returncode != 0:
            print(Path(f"/content/pose_shard{shard}.log").read_text(errors="ignore")[-3000:])
            raise RuntimeError(f"Shard {shard} lỗi (mã {process.returncode}); chạy lại cell để tiếp tục")
    print("Trích khung xương xong:", sum(1 for _ in POSE_ROOT.glob("*/*.npz")), "video")

## 8. Train ST-GCN
Luôn chạy với `--resume`: có checkpoint thì train tiếp, chưa có thì bắt đầu mới; lần train đã xong không bị train thêm khi chạy lại (xóa `checkpoint_last.pt` để train lại từ đầu).

In [ ]:
with step('8. Train'):
    run("scripts/train_fight.py", *dataset_args(DATASETS_TRAIN), "--pose-root", POSE_ROOT,
        "--output-dir", ARTIFACT_DIR, "--exclude", DEDUP_DIR / "exclude.json",
        "--epochs", 60, "--batch-size", 64, "--num-workers", NUM_WORKERS, "--device", DEVICE, "--resume")

## 9. Báo cáo training

In [ ]:
with step('9. Báo cáo training'):
    import matplotlib.pyplot as plt
    config = json.loads((ARTIFACT_DIR / "config.json").read_text())
    threshold = json.loads((ARTIFACT_DIR / "fight_threshold.json").read_text())
    history = json.loads((ARTIFACT_DIR / "training_history.json").read_text())["history"]
    for key in ["train_windows", "train_fight_windows", "val_windows", "parameter_count", "epochs_completed", "best_epoch", "best_val_auc"]:
        print(f"{key:22s}: {config[key]}")
    print("threshold:", threshold)

    epochs = [row["epoch"] for row in history]
    figure, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
    left.plot(epochs, [r["train_loss"] for r in history], label="train loss")
    left.plot(epochs, [r["val_loss"] for r in history], label="val loss")
    right.plot(epochs, [r["val_auc"] for r in history], label="val AUC")
    right.plot(epochs, [r["val_ap"] for r in history], label="val AP")
    for axis in (left, right):
        axis.axvline(config["best_epoch"], color="green", linestyle="--"); axis.grid(alpha=0.3); axis.legend(); axis.set_xlabel("Epoch")
    plt.show()

## 10. Đánh giá trên tập test
- UBI-Fights: AUC theo frame trên tập test rút gọn (40 video đánh nhau đầy đủ + 60 s mỗi video bình thường của `test_videos.csv`).
- Surveillance Camera Fight: test chéo theo clip.
- Nhóm `_crowd`: riêng các video trung bình ≥ 5 người/frame.
Ngưỡng lấy từ validation, không chỉnh theo nhãn test.

In [ ]:
with step('10. Đánh giá'):
    EVAL_DIR = ARTIFACT_DIR / "evaluation"
    run("scripts/evaluate_fight.py", *dataset_args(DATASETS_TEST), "--pose-root", POSE_ROOT,
        "--model", ARTIFACT_DIR / "fight_stgcn.pt", "--threshold", ARTIFACT_DIR / "fight_threshold.json",
        "--output-dir", EVAL_DIR, "--stride", EVAL_STRIDE, "--device", DEVICE)

## 11. Xem metrics và biểu đồ

In [ ]:
with step('11. Xem metrics'):
    from IPython.display import Image, display
    evaluation = json.loads((EVAL_DIR / "evaluation.json").read_text())
    print(f"Tập test rút gọn: {len(evaluation['shortened_test_videos'])} video bình thường lấy {TEST_NORMAL_SECONDS} s đầu; "
          "video đánh nhau giữ đầy đủ. Khi báo cáo, ghi rõ cách test này.")
    for level in ("frame_level", "clip_level"):
        for name, metrics in evaluation[level].items():
            print(f"\n{name} — {level}")
            for key in ["roc_auc", "average_precision", "f1", "precision", "recall", "accuracy", "balanced_accuracy"]:
                value = metrics[key]
                print(f"  {key:18s}: {value if value is None else round(value, 4)}")
            print(f"  confusion: TN={metrics['true_negative']} FP={metrics['false_positive']} FN={metrics['false_negative']} TP={metrics['true_positive']}")
    for image in sorted(EVAL_DIR.glob("*.png")):
        print(image.name); display(Image(filename=str(image)))

## 12. Đóng gói model
`fight_stgcn.pt` và `fight_threshold.json` đi cùng nhau; giải nén vào thư mục `models/` của repository. Bản ZIP luôn được lưu ở `MyDrive/SENTINEL/artifacts/fight/`; chỉ mở hộp thoại tải về khi `AUTO_DISCONNECT = False` (khi treo máy không có ai bấm).

In [ ]:
with step('12. Đóng gói model'):
    import zipfile
    from google.colab import files
    bundle = Path("/content/sentinel_fight_runtime.zip")
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
        for name in ["fight_stgcn.pt", "fight_threshold.json"]:
            archive.write(ARTIFACT_DIR / name, arcname=f"models/{name}")
        for name in ["config.json", "split.json", "training_history.json"]:
            archive.write(ARTIFACT_DIR / name, arcname=f"models/metadata/fight_{name}")
        archive.write(EVAL_DIR / "evaluation.json", arcname="models/metadata/fight_evaluation.json")
    import shutil
    shutil.copy2(bundle, ARTIFACT_DIR / bundle.name)
    print(ARTIFACT_DIR / bundle.name, f"{bundle.stat().st_size / 1024**2:.1f} MB")
    if not AUTO_DISCONNECT:
        files.download(str(bundle))

## 13. Kết thúc: ngắt runtime
Khi `AUTO_DISCONNECT = True`, cell này ghi "Hoàn tất" vào `MyDrive/SENTINEL/artifacts/fight/run_log.txt`, đồng bộ Drive rồi ngắt runtime để không tốn compute unit khi treo máy. Nếu một mục phía trên bị lỗi, runtime cũng tự ngắt và lý do được ghi vào cùng file log. Lần sau mở lại notebook: đọc `run_log.txt` để biết đã chạy tới đâu.

In [ ]:
finish_and_disconnect(f"Hoàn tất. Kết quả: {ARTIFACT_DIR / 'evaluation' / 'evaluation.json'}")